# Olist E-Commerce Data Cleaning

This notebook cleans and prepares the Olist datasets for business analysis and Power BI reporting.

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

In [4]:
DATA_PATH = Path("../data/raw/")

In [5]:
customers = pd.read_csv(DATA_PATH / "olist_customers_dataset.csv")
orders = pd.read_csv(DATA_PATH / "olist_orders_dataset.csv")
products = pd.read_csv(DATA_PATH / "olist_products_dataset.csv")
payments = pd.read_csv(DATA_PATH / "olist_order_payments_dataset.csv")
sellers = pd.read_csv(DATA_PATH / "olist_sellers_dataset.csv")
reviews = pd.read_csv(DATA_PATH / "olist_order_reviews_dataset.csv")
order_items = pd.read_csv(DATA_PATH / "olist_order_items_dataset.csv")
geolocation = pd.read_csv(DATA_PATH / "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(
    DATA_PATH / "product_category_name_translation.csv"
)

# Cleaning Orders

In [6]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [7]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"],
    errors="coerce"
)

In [8]:
orders["order_approved_at"] = pd.to_datetime(
    orders["order_approved_at"],
    errors="coerce"
)

In [9]:
orders["order_delivered_carrier_date"] = pd.to_datetime(
    orders["order_delivered_carrier_date"],
    errors="coerce"
)

In [10]:
orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"],
    errors="coerce"
)

In [11]:
orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"],
    errors="coerce"
)

In [12]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [13]:
orders[
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].isnull().sum()

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

### Orders Cleaning

- Converted all five order date columns from string to datetime.
- No additional missing values were created during conversion.
- Existing missing dates were retained because most relate to canceled or incomplete orders.
- Delivered orders with missing dates will be excluded only from date-dependent calculations.

# Cleaning Order-items

In [14]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


In [15]:
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

In [16]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 6.0 MB


In [18]:
order_items["shipping_limit_date"].isnull().sum()

np.int64(0)

### Order Items Cleaning

- Converted `shipping_limit_date` from string to datetime.
- No missing values were created during conversion.
- Zero freight values were retained because they may represent free shipping.
- No additional cleaning was required.

# Cleaning Payments

In [19]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [20]:
payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


In [21]:
payments["payment_type"].value_counts(dropna=False)

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [22]:
payments[
    payments["payment_type"] == "not_defined"
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [23]:
payments[
    payments["payment_value"] == 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [24]:
zero_payments = payments[
    payments["payment_value"] == 0
]

In [25]:
zero_payments.shape[0]

9

In [27]:
payments[
    payments["order_id"].isin(zero_payments["order_id"])
].sort_values(
    by=["order_id", "payment_sequential"]
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
33781,45ed6e85398a87c253db47c2d9f48216,1,voucher,1,21.13
11755,45ed6e85398a87c253db47c2d9f48216,2,voucher,1,50.01
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
40546,6ccb433e00daae1283ccc956189c82ae,1,credit_card,5,84.67
93478,6ccb433e00daae1283ccc956189c82ae,2,voucher,1,14.65
92318,6ccb433e00daae1283ccc956189c82ae,3,voucher,1,22.72
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
20963,8bcbe01d44d147f901cd3192671144db,1,credit_card,1,36.21


In [28]:
orders[
    orders["order_id"].isin(
        payments.loc[
            payments["payment_type"] == "not_defined",
            "order_id"
        ]
    )
][
    ["order_id", "order_status"]
]

,order_id,order_status
1130,00b1cb0320190ca0daa2c88b35206009,canceled
39919,4637ca194b6387e2d538dc89b124b0ee,canceled
40235,c8c528189310eaa44a745b8d9d26908b,canceled


In [29]:
payments = payments[
    payments["payment_value"] > 0
].copy()

In [30]:
payments["payment_type"].value_counts(dropna=False)

payment_type
credit_card    76795
boleto         19784
voucher         5769
debit_card      1529
Name: count, dtype: int64

In [31]:
(payments["payment_value"] == 0).sum()

np.int64(0)

In [32]:
payments[
    payments["payment_installments"] == 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [33]:
payments.loc[
    payments["payment_installments"] == 0,
    "payment_installments"
] = 1

In [34]:
payments["payment_installments"].min()

np.int64(1)

### Payments Cleaning

- Removed nine zero-value payment records.
- Six were zero-value voucher entries associated with orders that had other valid payments.
- Three were `not_defined` payments associated with canceled orders.
- Corrected two credit-card payments with zero installments by setting the installment count to one.
- No missing payment types remain.
Two positive credit-card payments had an installment count of zero. 
Because a single-payment transaction is represented as one installment, 
these values were corrected to one.

# Cleaning Reviews

In [35]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [36]:
reviews["review_creation_date"] = pd.to_datetime(
    reviews["review_creation_date"],
    errors="coerce"
)

In [37]:
reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"],
    errors="coerce"
)

In [38]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  str           
 1   order_id                 99224 non-null  str           
 2   review_score             99224 non-null  int64         
 3   review_comment_title     11568 non-null  str           
 4   review_comment_message   40977 non-null  str           
 5   review_creation_date     99224 non-null  datetime64[us]
 6   review_answer_timestamp  99224 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(1), str(4)
memory usage: 5.3 MB


In [39]:
reviews[
    [
        "review_creation_date",
        "review_answer_timestamp"
    ]
].isnull().sum()

review_creation_date       0
review_answer_timestamp    0
dtype: int64

In [40]:
reviews = (
    reviews
    .sort_values(
        by=["order_id", "review_answer_timestamp"]
    )
    .drop_duplicates(
        subset="order_id",
        keep="last"
    )
    .reset_index(drop=True)
)

In [43]:
reviews["order_id"].duplicated().sum()

np.int64(0)

In [44]:
reviews.shape

(98673, 7)

### Reviews Cleaning

- Converted the review date columns to datetime.
- Missing review titles and messages were retained because written comments were optional.
- For orders with multiple reviews, the latest review based on `review_answer_timestamp` was retained.
- Removed 551 additional review records.
- The cleaned table contains 98,673 unique order reviews.

# Cleaning Products

In [47]:
products.head(5)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [48]:
products = products.rename(
    columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length"
    }
)

In [49]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_length         32341 non-null  float64
 3   product_description_length  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


In [50]:
products[
    products["product_category_name"].isnull()
].isnull().sum()

product_id                      0
product_category_name         610
product_name_length           610
product_description_length    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64

In [51]:
products["product_category_name"] = (
    products["product_category_name"]
    .fillna("unknown")
)

In [53]:
products.isnull().sum()

product_id                      0
product_category_name           0
product_name_length           610
product_description_length    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [56]:
physical_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products[
    products[physical_columns].isnull().any(axis=1)
]

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [59]:
missing_physical_product_ids = products.loc[
    products[physical_columns].isnull().any(axis=1),
    "product_id"
]

order_items[
    order_items["product_id"].isin(
        missing_physical_product_ids
    )
][
    [
        "order_id",
        "product_id",
        "price",
        "freight_value"
    ]
]

,order_id,product_id,price,freight_value
7098,101157d4fae1c9fb74a00a5dee265c25,5eb564652db742ff8f28759cd8d2652a,29.0,14.52
9233,1521c6bb7b1028154c8c67cf80fa809f,5eb564652db742ff8f28759cd8d2652a,29.0,16.05
28715,415cfaaaa8cea49f934470548797fed1,5eb564652db742ff8f28759cd8d2652a,29.0,14.52
28716,415cfaaaa8cea49f934470548797fed1,5eb564652db742ff8f28759cd8d2652a,29.0,14.52
39299,595316a07cd3dea9db7adfcc7e247ae7,5eb564652db742ff8f28759cd8d2652a,39.0,9.27
48424,6e150190fbe04c642a9cf0b80d83ee16,5eb564652db742ff8f28759cd8d2652a,39.0,16.79
48980,6f497c40431d5fb0cfbd6c943dd29215,5eb564652db742ff8f28759cd8d2652a,29.0,10.96
58833,85f8ad45e067abd694b627859fa57453,09ff539a621711667c43eba6a3bd8466,1934.0,27.00
71134,a2456e7f02197951664897a94c87242d,5eb564652db742ff8f28759cd8d2652a,29.0,24.84
73556,a7a43f469c0d7bdb0a23a82db125aefa,5eb564652db742ff8f28759cd8d2652a,39.0,15.10


### Products Cleaning

* Corrected misspelled column names for product name and description length.
* Replaced 610 missing product categories with `unknown` to support category-level reporting.
* Missing values for product name length, description length, and photo quantity were retained because the original information was unavailable.
* Two products had missing weight and dimension values.
* These products were retained because they appeared in completed sales records.
* Missing physical attributes were not replaced with zero or estimated values to avoid introducing inaccurate data.
* Products with missing physical attributes will be excluded only from analyses that require weight or dimensions.


# Cleaning Category Translatiom

In [60]:
category_translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [61]:
category_translation.info()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB


In [62]:
category_translation[
    category_translation["product_category_name"] == "pc_gamer"
]

,product_category_name,product_category_name_english


In [63]:
category_translation[
    category_translation["product_category_name"] == "portateis_cozinha_e_preparadores_de_alimentos"
]

,product_category_name,product_category_name_english


In [64]:
additional_translations = pd.DataFrame({
    "product_category_name": [
        "portateis_cozinha_e_preparadores_de_alimentos",
        "pc_gamer",
        "unknown"
    ],
    "product_category_name_english": [
        "portable_kitchen_and_food_preparation_appliances",
        "pc_gamer",
        "unknown"
    ]
})

In [65]:
category_translation = pd.concat(
    [
        category_translation,
        additional_translations
    ],
    ignore_index=True
)

In [66]:
category_translation.tail()

,product_category_name,product_category_name_english
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes
70,seguros_e_servicos,security_and_services
71,portateis_cozinha_e_preparadores_de_alimentos,portable_kitchen_and_food_preparation_appliances
72,pc_gamer,pc_gamer
73,unknown,unknown


In [67]:
category_translation.shape

(74, 2)

In [68]:
category_translation[
    "product_category_name"
].duplicated().sum()

np.int64(0)

In [69]:
category_translation.isnull().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

In [70]:
products = products.merge(
    category_translation,
    how="left",
    on="product_category_name",
    validate="many_to_one"
)

In [71]:
products[
    "product_category_name_english"
].isnull().sum()

np.int64(0)

In [72]:
products.head()

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


### Category Translation Cleaning

* The original translation table contained 71 unique product categories with no missing or duplicate records.
* Added English translations for two product categories that were missing from the translation table.
* Added an `unknown` mapping for products whose original category was unavailable.
* The cleaned translation table contains 74 unique category mappings.
* Merged the English category name into the Products table using a validated many-to-one relationship.
* All products were retained during the merge, and no English category values remained missing.


# Cleaning Customers

In [73]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


In [74]:
customers["customer_zip_code_prefix"] = (
    customers["customer_zip_code_prefix"]
    .astype("string")
    .str.zfill(5)
)

In [75]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  str   
 1   customer_unique_id        99441 non-null  str   
 2   customer_zip_code_prefix  99441 non-null  string
 3   customer_city             99441 non-null  str   
 4   customer_state            99441 non-null  str   
dtypes: str(4), string(1)
memory usage: 3.8 MB


In [76]:
customers[
    "customer_zip_code_prefix"
].str.len().value_counts()

customer_zip_code_prefix
5    99441
Name: count, dtype: Int64

### Customers Cleaning

- Converted `customer_zip_code_prefix` from integer to string.
- Added leading zeros where necessary to standardize all ZIP-code prefixes to five characters.
- All 99,441 customer ZIP-code prefixes have a valid five-character format.
- No customer records were removed.

# Cleaning Sellers

In [77]:
sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB


In [78]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [79]:
sellers["seller_zip_code_prefix"] = (
    sellers["seller_zip_code_prefix"]
    .astype("string")
    .str.zfill(5)
)

In [80]:
sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   str   
 1   seller_zip_code_prefix  3095 non-null   string
 2   seller_city             3095 non-null   str   
 3   seller_state            3095 non-null   str   
dtypes: str(3), string(1)
memory usage: 96.8 KB


In [81]:
sellers[
    "seller_zip_code_prefix"
].str.len().value_counts()

seller_zip_code_prefix
5    3095
Name: count, dtype: Int64

In [82]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


### Sellers Cleaning

- Converted `seller_zip_code_prefix` from integer to string.
- Added leading zeros where necessary to standardize all ZIP-code prefixes to five characters.
- All 3,095 seller ZIP-code prefixes have a valid five-character format.
- No seller records were removed.

# Cleaning Geolocation

In [83]:
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [84]:
geolocation.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 38.2 MB


In [85]:
geolocation["geolocation_zip_code_prefix"] = geolocation["geolocation_zip_code_prefix"].astype("string").str.zfill(5)

In [86]:
geolocation.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  string 
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), str(2), string(1)
memory usage: 38.2 MB


In [87]:
geolocation[
    "geolocation_zip_code_prefix"
].str.len().value_counts()

geolocation_zip_code_prefix
5    1000163
Name: count, dtype: Int64

In [88]:
geolocation.duplicated().sum()

np.int64(261831)

In [89]:
geolocation.shape

(1000163, 5)

In [90]:
geolocation = (
    geolocation
    .drop_duplicates()
    .reset_index(drop=True)
)

In [91]:
geolocation.shape


(738332, 5)

In [92]:
outside_brazil = geolocation[
    (geolocation["geolocation_lat"] < -35) |
    (geolocation["geolocation_lat"] > 6) |
    (geolocation["geolocation_lng"] < -75) |
    (geolocation["geolocation_lng"] > -30)
]

In [93]:
outside_brazil.shape

(25, 5)

In [94]:
geolocation = geolocation[
    (geolocation["geolocation_lat"].between(-35, 6)) &
    (geolocation["geolocation_lng"].between(-75, -30))
].reset_index(drop=True)

In [95]:
geolocation.shape

(738307, 5)

In [96]:
(
    (~geolocation["geolocation_lat"].between(-35, 6)) |
    (~geolocation["geolocation_lng"].between(-75, -30))
).sum()

np.int64(0)

In [97]:
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.545621,-46.639292,sao paulo,SP
1,01046,-23.546081,-46.644820,sao paulo,SP
2,01046,-23.546129,-46.642951,sao paulo,SP
3,01041,-23.544392,-46.639499,sao paulo,SP
4,01035,-23.541578,-46.641607,sao paulo,SP


In [98]:
geolocation["geolocation_zip_code_prefix"].duplicated().sum()

np.int64(719296)

In [99]:
geolocation = (
    geolocation
    .groupby(
        "geolocation_zip_code_prefix",
        as_index=False
    )
    .agg(
        geolocation_lat=(
            "geolocation_lat",
            "mean"
        ),
        geolocation_lng=(
            "geolocation_lng",
            "mean"
        )
    )
)

In [100]:
geolocation.shape

(19011, 3)

In [101]:
geolocation[
    "geolocation_zip_code_prefix"
].duplicated().sum()

np.int64(0)

In [102]:
geolocation.isnull().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
dtype: int64

### Geolocation Cleaning

- Converted ZIP-code prefixes from integer to five-character strings.
- Removed 261,831 completely duplicate records.
- Removed 25 remaining coordinates outside the approximate geographic boundaries of Brazil.
- Grouped geographic records by ZIP-code prefix.
- Calculated the mean latitude and longitude for each ZIP-code prefix.
- Removed city and state columns because these attributes already exist in the Customers and Sellers tables.
- The cleaned table contains 19,011 unique ZIP-code prefixes with one coordinate record per prefix.

In [103]:
duplicate_checks = {
    "orders": orders["order_id"].duplicated().sum(),

    "order_items": order_items[
        ["order_id", "order_item_id"]
    ].duplicated().sum(),

    "payments": payments[
        ["order_id", "payment_sequential"]
    ].duplicated().sum(),

    "reviews": reviews["order_id"].duplicated().sum(),

    "products": products["product_id"].duplicated().sum(),

    "customers": customers["customer_id"].duplicated().sum(),

    "sellers": sellers["seller_id"].duplicated().sum(),

    "category_translation": category_translation[
        "product_category_name"
    ].duplicated().sum(),

    "geolocation": geolocation[
        "geolocation_zip_code_prefix"
    ].duplicated().sum()
}

In [104]:
pd.Series(
    duplicate_checks,
    name="duplicate_count"
)

orders                  0
order_items             0
payments                0
reviews                 0
products                0
customers               0
sellers                 0
category_translation    0
geolocation             0
Name: duplicate_count, dtype: int64

In [105]:
relationship_checks = {
    "orders_without_customer": (
        ~orders["customer_id"].isin(
            customers["customer_id"]
        )
    ).sum(),

    "items_without_order": (
        ~order_items["order_id"].isin(
            orders["order_id"]
        )
    ).sum(),

    "items_without_product": (
        ~order_items["product_id"].isin(
            products["product_id"]
        )
    ).sum(),

    "items_without_seller": (
        ~order_items["seller_id"].isin(
            sellers["seller_id"]
        )
    ).sum(),

    "payments_without_order": (
        ~payments["order_id"].isin(
            orders["order_id"]
        )
    ).sum(),

    "reviews_without_order": (
        ~reviews["order_id"].isin(
            orders["order_id"]
        )
    ).sum(),

    "products_without_translation": (
        ~products["product_category_name"].isin(
            category_translation["product_category_name"]
        )
    ).sum()
}

In [106]:
pd.Series(
    relationship_checks,
    name="orphan_count"
)

orders_without_customer         0
items_without_order             0
items_without_product           0
items_without_seller            0
payments_without_order          0
reviews_without_order           0
products_without_translation    0
Name: orphan_count, dtype: int64

In [107]:
geolocation_coverage = {
    "customer_zips_without_geolocation": (
        ~customers["customer_zip_code_prefix"].isin(
            geolocation["geolocation_zip_code_prefix"]
        )
    ).sum(),

    "seller_zips_without_geolocation": (
        ~sellers["seller_zip_code_prefix"].isin(
            geolocation["geolocation_zip_code_prefix"]
        )
    ).sum()
}

In [108]:
pd.Series(
    geolocation_coverage,
    name="missing_geolocation_count"
)

customer_zips_without_geolocation    279
seller_zips_without_geolocation        7
Name: missing_geolocation_count, dtype: int64

# Saving Clean Datasets


In [109]:
PROCESSED_PATH = Path("../data/processed/")

PROCESSED_PATH.mkdir(
    parents=True,
    exist_ok=True
)

In [110]:
cleaned_tables = {
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "customers": customers,
    "sellers": sellers,
    "category_translation": category_translation,
    "geolocation": geolocation
}

In [111]:
for table_name, dataframe in cleaned_tables.items():
    dataframe.to_csv(
        PROCESSED_PATH / f"{table_name}_clean.csv",
        index=False
    )

In [112]:
sorted(PROCESSED_PATH.glob("*.csv"))

[WindowsPath('../data/processed/category_translation_clean.csv'),
 WindowsPath('../data/processed/customers_clean.csv'),
 WindowsPath('../data/processed/geolocation_clean.csv'),
 WindowsPath('../data/processed/order_items_clean.csv'),
 WindowsPath('../data/processed/orders_clean.csv'),
 WindowsPath('../data/processed/payments_clean.csv'),
 WindowsPath('../data/processed/products_clean.csv'),
 WindowsPath('../data/processed/reviews_clean.csv'),
 WindowsPath('../data/processed/sellers_clean.csv')]